# 10: Automated Mask Generation from Eye Tracking

This notebook implements a fully automated pipeline for generating SAM2 segmentation masks from Project Aria eye-tracking data. Instead of manually selecting frames and drawing bounding boxes as prompts, we use the gaze signal recorded by the Aria glasses to automatically identify which objects the wearer is looking at.

### In this notebook, we will:
1. **Recording setup**: Configure paths for the VRS recording and its MPS outputs.
2. **Gaze loading & alignment**: Load the MPS eye-gaze stream and align it temporally with the RGB frames.
3. **Gaze projection**: Convert the raw gaze direction into a 2D pixel coordinate on the RGB image plane.
4. **Fixation detection**: Apply a dispersion-based fixation detector (I-DT) to identify sustained gaze episodes that indicate intentional object observation.
5. **SAM2 prompt generation**: Use each fixation centroid as a point prompt for SAM2, with deduplication to avoid re-segmenting already-masked objects.
6. **Mask export & validation**: Save the generated masks with diagnostic overlays.
7. **Conclusions & handoff**: Summary of detected objects and readiness for the NeRF reconstruction pipeline.

**Note**:  The Aria ET camera operates at ~10 fps while the RGB camera runs at 30 fps. The MPS eye-gaze output inherits the ET rate, so temporal alignment is required before any gaze-to-image projection.

As a sample, we will use the `office_cabinet.vrs` recording

## 10.1 Recording Configuration & Setup

Configure all input/output paths for the target recording.

In [1]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from PIL import Image
from projectaria_tools.core import data_provider, mps

RECORDING_NAME = "office_cabinet"

# input paths
DATA_ROOT = os.path.join("..", "data", "raw", RECORDING_NAME)
VRS_PATH = os.path.join(DATA_ROOT, f"{RECORDING_NAME}.vrs")
MPS_ROOT = os.path.join(DATA_ROOT, f"mps_{RECORDING_NAME}_vrs")
SLAM_DIR = os.path.join(MPS_ROOT, "slam")
GAZE_DIR = os.path.join(MPS_ROOT, "eye_gaze")
GAZE_CSV = os.path.join(GAZE_DIR, "general_eye_gaze.csv")

# output paths
OUT_ROOT = os.path.join("..", "data", "outputs", "segmentation", RECORDING_NAME)
FRAMES_DIR = os.path.join(OUT_ROOT, "frames_10fps")
MASKS_DIR = os.path.join(OUT_ROOT, "sam2", "masks")

for d in [FRAMES_DIR, MASKS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"Recording: {RECORDING_NAME}")
print(f"VRS: {os.path.abspath(VRS_PATH)}")
print(f"Gaze CSV: {GAZE_CSV}")
print(f"MPS root: {os.path.abspath(MPS_ROOT)}")
print(f"Output: {os.path.abspath(OUT_ROOT)}")

Recording : office_cabinet
VRS       : c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\raw\office_cabinet\office_cabinet.vrs
MPS root  : c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\raw\office_cabinet\mps_office_cabinet_vrs
Output    : c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\segmentation\office_cabinet


### 10.1.2 VRS Access & RGB Stream Resolution

Open the VRS file, resolve the RGB stream, and read its metadata. This validates that the recording is decodable and provides the frame count and resolution needed for downstream extraction.

In [4]:
# Open VRS and resolve RGB stream

provider = data_provider.create_vrs_data_provider(VRS_PATH)

rgb_stream_id = provider.get_stream_id_from_label("camera-rgb")
rgb_config    = provider.get_image_configuration(rgb_stream_id)
n_rgb_total   = provider.get_num_data(rgb_stream_id)

IMG_W = rgb_config.image_width
IMG_H = rgb_config.image_height

print(f"RGB stream ID: {rgb_stream_id}")
print(f"RGB frames: {n_rgb_total}")
print(f"Resolution: {IMG_W}x{IMG_H}")

RGB stream ID: 214-1
RGB frames: 1207
Resolution: 1408x1408


### 10.1.3 RGB Frame Extraction at 10 fps

The native RGB stream runs at 30 fps. We subsample every 3rd frame to produce a 10 fps frame sequence, matching the rate of the eye-tracking stream. Frames are saved as JPEG images with 0-indexed filenames `000000.jpg`, `000001.jpg`, etc. in the `frames_10fps/` directory.

In [5]:
# Helper to extract image array from a projectaria_tools sensor sample

from operator import index

from matplotlib.pylab import sample


def extract_rgb_frame(provider, stream_id, index):
    """Read one RGB frame from the VRS by index, return as HxWx3 numpy array."""
    sample = provider.get_sensor_data_by_index(stream_id, index)

# Navigate the accessor hierarchy
    for attr in ["image_data_and_record", "image_data", "image"]:
        if not hasattr(sample, attr):
            continue
        payload = getattr(sample, attr)
        payload = payload() if callable(payload) else payload

        # Handle (ImageData, record) tuples
        if isinstance(payload, tuple):
            payload = payload[0]

        # Try .to_numpy_array()
        for arr_attr in ["to_numpy_array", "buffer"]:
            if hasattr(payload, arr_attr):
                arr = getattr(payload, arr_attr)
                arr = arr() if callable(arr) else arr
                arr = np.asarray(arr)
                if arr.ndim >= 2:
                    # Channel-first to channel-last
                    if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
                        arr = np.transpose(arr, (1, 2, 0))
                    return arr.astype(np.uint8)

    raise RuntimeError(f"Cannot extract image from sample at index {index}")


# Validate extraction on a single frame
test_frame = extract_rgb_frame(provider, rgb_stream_id, 0)
print(f"Frame extraction OK: shape={test_frame.shape}, dtype={test_frame.dtype}")

Frame extraction OK: shape=(1408, 1408, 3), dtype=uint8


In [14]:
#Extract every 3rd frame (30 fps → 10 fps) and save to disk

SUBSAMPLE_STRIDE = 3  # 30 fps / 3 = 10 fps

indices_10fps = list(range(0, n_rgb_total, SUBSAMPLE_STRIDE))
n_frames_10fps = len(indices_10fps)

print(f"Subsampling: stride={SUBSAMPLE_STRIDE} to {n_frames_10fps} frames at 10 fps")

existing = len([f for f in os.listdir(FRAMES_DIR) if f.endswith(".jpg")])
if existing >= n_frames_10fps:
    print(f"Frames already extracted ({existing} files), skipping.")
else:
    for out_idx, vrs_idx in enumerate(indices_10fps):
        frame = extract_rgb_frame(provider, rgb_stream_id, vrs_idx)
        pil_img = Image.fromarray(frame)
        out_path = os.path.join(FRAMES_DIR, f"{out_idx:06d}.jpg")
        pil_img.save(out_path, quality=95)
        if (out_idx + 1) % 50 == 0:
            print(f"  [{out_idx+1}/{n_frames_10fps}] saved")

print(f"{n_frames_10fps} frames saved to: {FRAMES_DIR}")

print(f"{n_frames_10fps} frames saved to: {FRAMES_DIR}")

Subsampling: stride=3 to 403 frames at 10 fps
  [50/403] saved
  [100/403] saved
  [150/403] saved
  [200/403] saved
  [250/403] saved
  [300/403] saved
  [350/403] saved
  [400/403] saved
403 frames saved to: ..\data\outputs\segmentation\office_cabinet\frames_10fps
403 frames saved to: ..\data\outputs\segmentation\office_cabinet\frames_10fps


In [8]:
# Build timestamp arrays for both the full 30fps stream and the 10fps subset

from projectaria_tools.core.sensor_data import TimeDomain

rgb_timestamps_ns_30fps = np.array([
                        provider.get_sensor_data_by_index(rgb_stream_id, i).get_time_ns(TimeDomain.DEVICE_TIME)
                        for i in range(n_rgb_total)], dtype=np.int64)

# 10fps subset timestamps
rgb_timestamps_ns = rgb_timestamps_ns_30fps[indices_10fps]

rgb_duration_s = (rgb_timestamps_ns[-1] - rgb_timestamps_ns[0]) / 1e9
rgb_rate_hz = len(rgb_timestamps_ns) / rgb_duration_s

print(f"10fps frames: {len(rgb_timestamps_ns)}")
print(f"Duration: {rgb_duration_s:.2f} s")
print(f"Measured rate: {rgb_rate_hz:.2f} Hz")

10fps frames   : 403
Duration       : 40.19 s
Measured rate   : 10.03 Hz


## 10.2 Eye Gaze Loading & Temporal Alignment

Load the MPS eye-gaze stream and align it with the extracted 10 fps RGB frames. The gaze data is sampled by the ET cameras at a rate distinct from the RGB camera. For each 10 fps frame, we find the temporally closest gaze sample via searchsorted and flag matches that exceed a configurable time-delta threshold.

In [9]:
# Load eye-gaze samples via the MPS API

eye_gazes = mps.read_eyegaze(GAZE_CSV)
n_gaze = len(eye_gazes)

if n_gaze == 0:
    raise ValueError("Eye-gaze CSV is empty, check MPS processing.")

print(f"Gaze samples loaded: {n_gaze}")

# Inspect available fields on the first sample for debugging

sample_fields = [a for a in dir(eye_gazes[0]) if not a.startswith("_")]
print(f"Sample fields: {sample_fields}")

Gaze samples loaded: 402
Sample fields: ['depth', 'pitch', 'pitch_high', 'pitch_low', 'session_uid', 'tracking_timestamp', 'vergence', 'yaw', 'yaw_high', 'yaw_low']


In [10]:
#Extract gaze timestamps in nanoseconds
# The exact accessor varies by projectaria_tools version

g0 = eye_gazes[0]

if hasattr(g0, "tracking_timestamp") and hasattr(g0.tracking_timestamp, "total_seconds"):
    # timedelta object
    gaze_timestamps_ns = np.array([int(g.tracking_timestamp.total_seconds() * 1e9) for g in eye_gazes], dtype=np.int64)
    print("Timestamp accessor: tracking_timestamp.total_seconds()")

elif hasattr(g0, "tracking_timestamp_us"):
    # integer microseconds
    gaze_timestamps_ns = np.array([g.tracking_timestamp_us * 1000 for g in eye_gazes], dtype=np.int64)
    print("Timestamp accessor: tracking_timestamp_us")

else:
    raise AttributeError(f"Cannot determine timestamp field. Available: {sample_fields}")

gaze_duration_s = (gaze_timestamps_ns[-1] - gaze_timestamps_ns[0]) / 1e9
gaze_rate_hz = n_gaze / gaze_duration_s

print(f"Gaze duration: {gaze_duration_s:.2f} s")
print(f"Gaze rate: {gaze_rate_hz:.2f} Hz")
print(f"Rate mismatch: RGB {rgb_rate_hz:.1f} Hz vs Gaze {gaze_rate_hz:.1f} Hz")

Timestamp accessor: tracking_timestamp.total_seconds()
Gaze duration: 40.09 s
Gaze rate: 10.03 Hz
Rate mismatch: RGB 10.0 Hz vs Gaze 10.0 Hz


In [11]:
# Extract yaw and pitch fields (CPF = Central Pupil Frame)

g0 = eye_gazes[0]

if hasattr(g0, "yaw"):
    get_yaw   = lambda g: g.yaw
    get_pitch = lambda g: g.pitch
    print("Gaze direction fields: yaw, pitch")
elif hasattr(g0, "yaw_rads_cpf"):
    get_yaw   = lambda g: g.yaw_rads_cpf
    get_pitch = lambda g: g.pitch_rads_cpf
    print("Gaze direction fields: yaw_rads_cpf, pitch_rads_cpf")
else:
    raise AttributeError(f"Cannot find yaw/pitch fields. Available: {sample_fields}")

# Sanity check on gaze range
yaw_all   = np.array([get_yaw(g) for g in eye_gazes])
pitch_all = np.array([get_pitch(g) for g in eye_gazes])

print(f"Yaw range: [{np.degrees(yaw_all.min()):.1f}, {np.degrees(yaw_all.max()):.1f}] deg")
print(f"Pitch range: [{np.degrees(pitch_all.min()):.1f}, {np.degrees(pitch_all.max()):.1f}] deg")

Gaze direction fields: yaw, pitch
Yaw   range: [-48.3, 40.2] deg
Pitch range: [-42.5, 19.2] deg


In [12]:
# Temporal alignment, for each 10fps RGB frame, find the nearest gaze sample.

MAX_DELTA_MS = 60.0 # half the gaze interval at 10 fps

# Nearest-neighbour matching via binary search

insert_pos = np.searchsorted(gaze_timestamps_ns, rgb_timestamps_ns)
insert_pos = np.clip(insert_pos, 1, n_gaze - 1)

delta_before = np.abs(rgb_timestamps_ns - gaze_timestamps_ns[insert_pos - 1])
delta_after  = np.abs(rgb_timestamps_ns - gaze_timestamps_ns[insert_pos])

nearest_gaze_idx = np.where(delta_before < delta_after, insert_pos - 1, insert_pos)
nearest_delta_ms = np.minimum(delta_before, delta_after) / 1e6

reliable = nearest_delta_ms < MAX_DELTA_MS

print(f"Frames aligned: {len(rgb_timestamps_ns)}")
print(f"Reliable: {reliable.sum()} ({100*reliable.mean():.1f}%)")
print(f"Median delta: {np.median(nearest_delta_ms):.2f} ms")
print(f"Max delta: {nearest_delta_ms.max():.2f} ms")
print(f"Unreliable: {(~reliable).sum()}")

Frames aligned: 403
Reliable: 402 (99.8%)
Median delta: 33.22 ms
Max delta: 66.76 ms
Unreliable: 1


In [13]:
# Build the final alignment table: one row per 10fps frame

alignment = pd.DataFrame({
"frame_idx":     np.arange(len(rgb_timestamps_ns)),
"rgb_ts_ns":     rgb_timestamps_ns,
"gaze_idx":      nearest_gaze_idx,
"gaze_ts_ns":    gaze_timestamps_ns[nearest_gaze_idx],
"delta_ms":      nearest_delta_ms,
"reliable":      reliable,
"yaw_rads":      [get_yaw(eye_gazes[i])   for i in nearest_gaze_idx],
"pitch_rads":    [get_pitch(eye_gazes[i]) for i in nearest_gaze_idx],
})

print(alignment.head(10).to_string(index=False))
print(f"\n... ({len(alignment)} rows)")
print(f"\nAlignment table ready. {reliable.sum()}/{len(alignment)} frames have reliable gaze data.")

 frame_idx      rgb_ts_ns  gaze_idx     gaze_ts_ns  delta_ms  reliable  yaw_rads  pitch_rads
         0 11201333999362         0 11201400756000 66.756638     False  0.181250   -0.139721
         1 11201435360612         0 11201400756000 34.604612      True  0.181250   -0.139721
         2 11201533968612         1 11201500740000 33.228612      True  0.144844   -0.138911
         3 11201633950787         2 11201600724000 33.226787      True  0.108724   -0.127414
         4 11201733790450         3 11201700708000 33.082450      True  0.151358   -0.064705
         5 11201833915075         4 11201800692000 33.223075      True  0.137418   -0.077216
         6 11201933901537         5 11201900676000 33.225537      True  0.149647   -0.081064
         7 11202033883412         6 11202000660000 33.223412      True -0.012079   -0.174748
         8 11202133867825         7 11202100644000 33.223825      True  0.045768   -0.130679
         9 11202233856612         8 11202200628000 33.228612      True

## 10.3 Gaze Projection to 2D Image Plane

To use the eye-tracking data as prompts for SAM2, we must project the 3D gaze vector onto the 2D pixel coordinate system of the RGB camera.

The Aria gaze data is defined in the Central Pupil Frame (CPF). The mathematical transformation chain is:
`CPF Frame` to`Device Frame` to`RGB Camera Frame` to `2D Pixel Coordinates`

First, we load the specific device calibration from the VRS provider.

In [15]:
# Extract device and camera calibration
device_calib = provider.get_device_calibration()
rgb_calib = device_calib.get_camera_calib("camera-rgb")

# Extract the spatial transformations (Sophus SE3 / RigidTransform)
T_device_cpf = device_calib.get_transform_device_cpf()
T_device_rgb = rgb_calib.get_transform_device_camera()

# Invert to get Camera <- Device
T_rgb_device = T_device_rgb.inverse()

print(f"RGB Camera Intrinsics: {rgb_calib.get_model_name()}")
print(f"Calibration loaded successfully.")

RGB Camera Intrinsics: CameraModelType.FISHEYE624
Calibration loaded successfully.


### 10.3.1 Defining the Projection Logic

We define a dedicated function to perform the ray transformations. The gaze direction is represented by `yaw` and `pitch` angles. We compute the directional vector at an arbitrary depth since the 2D projection is scale-invariant.

In [16]:
def project_gaze_to_rgb(yaw_rads, pitch_rads, rgb_calib, T_device_cpf, T_rgb_device):
    # Define the 3D gaze vector in the CPF frame, assuming standard pinhole projection model for the optical ray
    gaze_vector_cpf = np.array([np.tan(yaw_rads), np.tan(pitch_rads), 1.0])

    # Transform the vector to the RGB camera coordinate system
    gaze_vector_device = T_device_cpf @ gaze_vector_cpf
    gaze_vector_rgb = T_rgb_device @ gaze_vector_device
    
    # Project to 2D pixel space
    pixel = rgb_calib.project(gaze_vector_rgb)
    
    return pixel

### 10.3.2 Applying Projection to the Aligned Dataset

We iterate over the temporally aligned gaze dataframe (`df_gaze_aligned`) to compute the 2D coordinates for each synchronized frame. We also filter out projections that fall completely outside the RGB sensor's Field of View.

In [ ]:
# Initialize empty lists for coordinates
gaze_pixels_x = []
gaze_pixels_y = []
valid_flags = []

# Apply projection frame-by-frame
for idx, row in df_gaze_aligned.iterrows():
    # Handle NaN values explicitly
    if pd.isna(row['yaw']) or pd.isna(row['pitch']):
        gaze_pixels_x.append(np.nan)
        gaze_pixels_y.append(np.nan)
        valid_flags.append(False)
        continue
        
    pixel = project_gaze_to_rgb(row['yaw'], row['pitch'], rgb_calib, T_device_cpf, T_rgb_device)
    
    if pixel is not None:
        u, v = pixel[0], pixel[1]
        
        # FOV Boundary Check
        is_valid = (0 <= u < IMG_W) and (0 <= v < IMG_H)
        
        gaze_pixels_x.append(u)
        gaze_pixels_y.append(v)
        valid_flags.append(is_valid)
    else:
        gaze_pixels_x.append(np.nan)
        gaze_pixels_y.append(np.nan)
        valid_flags.append(False)

# Store results in the dataframe
df_gaze_aligned['gaze_x'] = gaze_pixels_x
df_gaze_aligned['gaze_y'] = gaze_pixels_y
df_gaze_aligned['valid_gaze'] = valid_flags

valid_count = df_gaze_aligned['valid_gaze'].sum()
total_count = len(df_gaze_aligned)

print(f"Projection complete. Valid in-frame gaze points: {valid_count}/{total_count}")
df_gaze_aligned[['timestamp_rgb', 'gaze_x', 'gaze_y', 'valid_gaze']].head()